# Exp 02: Grad-CAM

Card: [exp02_grad_cam.md](../../experiments/exp02_grad_cam.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow

from core.config import ROOT
from core.cv import VIP, Config
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    EXPERIMENT = "cnn-model-grad-cam-smoke"
    N_SHOW = 2  # images per figure
else:
    EXPERIMENT = "cnn-model-grad-cam"
    N_SHOW = 4
CONFIG = Config.from_toml(
    ROOT / "sections" / "model" / "configs" / "exp01_lecture_models" / "resnet18.toml"
)
FIGURES = ROOT / "sections" / "model" / "figures"
SAVE_FIGURES = not SMOKE

plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.titlesize": "small",
        "axes.titlecolor": "#52514e",
    }
)


## Predictions and fold models


In [ ]:
cams = GradCamFigures(CONFIG)

## Examples per VIP class


In [ ]:
figures = {
    f"exp02_grad_cam_{cls.lower()}_{outcome}.png": fig
    for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items()
}
plt.show()

In [ ]:
with Experiment(EXPERIMENT).run("grad-cam", {"model": CONFIG.name, "folds": CONFIG.folds, "n_show": N_SHOW}) as run:
    mlflow.log_metrics({f"threshold_{c}": cams.thresholds[c] for c in VIP})
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)